In [ ]:
"""
APM_52065_EP — Crypto Flash Crash October 2025
STEP 0: Data Download and Preparation
 
To run before running section3_time_series.py
 

    
"""
import pip


pip install yfinance pandas requests
import os
import time
import requests
import pandas as pd
import yfinance as yf
 
os.makedirs("data", exist_ok=True)

In [4]:
# ─────────────────────────────────────────────────────────────
# SOURCE 1: Yahoo Finance — BTC & ETH hourly, full year 2025
# ─────────────────────────────────────────────────────────────
def download_yahoo_hourly():
    """
    yfinance allows a maximum of 730 days at hourly frequency.
    We split into two semesters to cover the full year 2025.
    """
    tickers = {"BTC": "BTC-USD", "ETH": "ETH-USD"}
    frames = {}
 
    for name, ticker in tickers.items():
        print(f"[Yahoo] Downloading {ticker} hourly data 2025...")
        # First semester
        s1 = yf.download(ticker, start="2025-01-01", end="2025-07-01",
                         interval="1h", progress=False)["Close"]
        # Second semester
        s2 = yf.download(ticker, start="2025-07-01", end="2026-01-01",
                         interval="1h", progress=False)["Close"]
 
        combined = pd.concat([s1, s2]).sort_index()
        combined = combined[~combined.index.duplicated()]
        combined.name = name
        frames[name] = combined
 
        path = f"data/{name}_USD_1h_2025.csv"
        combined.to_csv(path, header=True)
        print(f"  -> {len(combined)} observations saved to {path}")
 
    return frames["BTC"], frames["ETH"]
 
 


In [5]:
# ─────────────────────────────────────────────────────────────
# SOURCE 2: Binance Public API — 1-minute data, October 2025
#           (used for Hawkes process analysis in Section 6)
# ─────────────────────────────────────────────────────────────
def download_binance_minute(symbol="BTCUSDT", start_ms=None,
                            end_ms=None, limit=1000):
    """
    Downloads Binance klines (OHLCV) at 1-minute frequency.
    No API key required for public market data.
    """
    url = "https://api.binance.com/api/v3/klines"
    if start_ms is None:
        start_ms = int(
            pd.Timestamp("2025-10-06 00:00:00", tz="UTC").timestamp() * 1000
        )
    if end_ms is None:
        end_ms = int(
            pd.Timestamp("2025-10-13 00:00:00", tz="UTC").timestamp() * 1000
        )
 
    all_rows = []
    current = start_ms
 
    while current < end_ms:
        params = {
            "symbol":    symbol,
            "interval":  "1m",
            "startTime": current,
            "endTime":   end_ms,
            "limit":     limit,
        }
        try:
            resp = requests.get(url, params=params, timeout=10)
            resp.raise_for_status()
            rows = resp.json()
            if not rows:
                break
            all_rows.extend(rows)
            current = rows[-1][0] + 60_000   # advance by one minute
            time.sleep(0.2)                   # respect Binance rate limit
        except Exception as e:
            print(f"  [WARN] Binance API error: {e}")
            break
 
    if not all_rows:
        print(f"  [WARN] No data received for {symbol}.")
        return pd.DataFrame()
 
    df = pd.DataFrame(all_rows, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_vol", "trades", "taker_buy_base",
        "taker_buy_quote", "ignore",
    ])
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("time")[["close", "volume"]].astype(float)
    df.columns = [f"{symbol}_close", f"{symbol}_volume"]
    return df
 
 
def download_all_binance_minute():
    print("[Binance] Downloading BTC & ETH at 1-min frequency, Oct 6–12 2025...")
    btc_1m = download_binance_minute("BTCUSDT")
    eth_1m = download_binance_minute("ETHUSDT")
 
    if not btc_1m.empty and not eth_1m.empty:
        merged = btc_1m.join(eth_1m, how="outer")
        merged.to_csv("data/BTCETH_USDT_1m_oct2025.csv")
        print(f"  -> {len(merged)} rows saved to data/BTCETH_USDT_1m_oct2025.csv")
    else:
        print("  [WARN] Incomplete Binance data — check network connection.")
    return btc_1m, eth_1m
 
 


In [6]:
# ─────────────────────────────────────────────────────────────
# SOURCE 3: Yahoo Finance — S&P 500 and VIX daily 2025
# ─────────────────────────────────────────────────────────────
def download_macro():
    print("[Yahoo] Downloading SPX & VIX daily data 2025...")
    spx = yf.download("^GSPC", start="2025-01-01", end="2026-01-01",
                      interval="1d", progress=False)["Close"]
    vix = yf.download("^VIX",  start="2025-01-01", end="2026-01-01",
                      interval="1d", progress=False)["Close"]
    spx.to_csv("data/SPX_1d_2025.csv", header=True)
    vix.to_csv("data/VIX_1d_2025.csv", header=True)
    print(f"  -> SPX: {len(spx)} obs.  |  VIX: {len(vix)} obs.")
    return spx, vix
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOCAL LOADING 
# ─────────────────────────────────────────────────────────────
def load_local():
    btc = pd.read_csv("data/BTC_USD_1h_2025.csv",
                      index_col=0, parse_dates=True).squeeze()
    eth = pd.read_csv("data/ETH_USD_1h_2025.csv",
                      index_col=0, parse_dates=True).squeeze()
    return btc, eth
 
 


In [8]:
# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    btc_h, eth_h = download_yahoo_hourly()
    btc_1m, eth_1m = download_all_binance_minute()
    spx, vix = download_macro()
 
    print("\n── Data summary ─────────────────────────────────────")
    print(f"BTC hourly  : {btc_h.index[0]} -> {btc_h.index[-1]}  ({len(btc_h)} obs.)")
    print(f"ETH hourly  : {eth_h.index[0]} -> {eth_h.index[-1]}  ({len(eth_h)} obs.)")
    print(f"SPX daily   : {len(spx)} obs.")
    print(f"VIX daily   : {len(vix)} obs.")
    print("\n[DONE] All data saved in ./data/")

[Yahoo] Downloading BTC-USD hourly data 2025...
  -> 8604 observations saved to data/BTC_USD_1h_2025.csv
[Yahoo] Downloading ETH-USD hourly data 2025...
  -> 8604 observations saved to data/ETH_USD_1h_2025.csv
[Binance] Downloading BTC & ETH at 1-min frequency, Oct 6–12 2025...
  -> 10081 rows saved to data/BTCETH_USDT_1m_oct2025.csv
[Yahoo] Downloading SPX & VIX daily data 2025...
  -> SPX: 250 obs.  |  VIX: 250 obs.

── Data summary ─────────────────────────────────────
BTC hourly  : 2025-01-01 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00  (8604 obs.)
ETH hourly  : 2025-01-01 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00  (8604 obs.)
SPX daily   : 250 obs.
VIX daily   : 250 obs.

[DONE] All data saved in ./data/
